# WorldQuant BRAIN Alpha 遍历 V2.1

目标：**优先提高可提交 Alpha 的产出效率**。默认研究模式为 `POWER_POOL_ATOM`，同时保留 `ATOM / POWER_POOL / REGULAR`。

主流程：Data Fields → Data Coverage Filter → Stage1 Core → Targeted Extension → Stage2 Neutralize → Group-Rank Fallback → Live Check → Fail-aware Repair → Final Check。

V2.1 原则：
- 不改变 `simulation_key = expression + settings`，继续复用原 `alpha_results.db`。
- 不写死 Power Pool 支持地区；平台实时 eligibility/check 为最终依据。
- Data Coverage 阈值由 Notebook 配置，默认 `>= 0.90`。
- GLB / 其他地区的并发上限也由 Notebook 配置，默认分别 4 / 8。
- 不加入 Dataset Historical Coverage Preflight。


## 1. 环境初始化

加载 V2.1 库。请把 `machine_lib_V2_1.py`、本 Notebook 和原 `alpha_results.db` 放在同一项目目录。


In [1]:
# 加载依赖和 V2.1 模块，并确认当前 Python 与项目目录。
import sys
import importlib
from pathlib import Path

import pandas as pd
import machine_lib_V2_1 as machine_lib

# 重新加载本地库，确保刚修改的 .py 立即生效。
machine_lib = importlib.reload(machine_lib)
from machine_lib_V2_1 import *

# 记录项目路径，并统一设置结果表的显示宽度。
module_path = Path(machine_lib.__file__).resolve()
project_dir = module_path.parent
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 50)

print("V2.1 library:", module_path)
print("Project:", project_dir)
print("Python:", sys.executable)


V2.1 library: F:\一二三\v2\machine_lib_V2_1.py
Project: F:\一二三\v2
Python: c:\Users\Administrator\AppData\Local\Programs\Python\Python39\python.exe


## 2. 参数配置

只需要优先检查这一格。所有阈值和并发限制都在这里配置，不隐藏在 Notebook 逻辑里。


In [2]:
# 本格集中配置研究目标、BRAIN 参数、字段筛选、并发和各阶段开关。
# =========================
# 研究目标
# =========================
# 选择本轮主要目标模式；默认优先寻找 PP + ATOM。
TARGET_MODE = "POWER_POOL_ATOM"   # POWER_POOL_ATOM / ATOM / POWER_POOL / REGULAR

# =========================
# BRAIN 回测设置
# =========================
# 设置本轮 BRAIN 回测环境；Power Pool 支持地区不在本地写死。
REGION = "GLB"                    # 不在本地写死 Power Pool 支持地区
UNIVERSE = "TOPDIV3000"
DELAY = 1
DATASET_ID = "risk60"
NEUTRALIZATION = "SUBINDUSTRY"
INIT_DECAY = 4
TRUNCATION = 0.08
TEST_PERIOD = "P0Y"

# =========================
# 数据字段筛选
# =========================
# 只让达到 Coverage 阈值的字段进入正式搜索，阈值可自行调整。
MIN_DATA_COVERAGE = 0.90           # 可改；>= 该值才进入 Stage1
DATA_COVERAGE_COLUMN = None        # None=自动识别 API 中真正有效的 coverage 列
MAX_FIELDS = None                  # 在 Coverage + 人工排除之后再截取

# 人工排除只是额外保护，不等同于 Dataset 历史覆盖预检。
MANUAL_EXCLUDED_FIELDS_BY_DATASET = {
    "other169": {
        "bid_price_percent_of_notional",
        "interest_rate_curve_parallel_shift_impact",
        "isda_assumed_recovery_percent",
        "mid_par_spread_basis_points",
        "mid_quote_spread_basis_points",
        "oth169_monthlyobservationscore",
        "oth169_percentofparoffer",
        "oth169_recrisk",
    },
}

# =========================
# 并发与轮询设置
# =========================
# 设置期望并发数；底层会按地区上限自动收紧。
CONCURRENCY = 4                    # 你希望开的并发数
GLB_MAX_CONCURRENCY = 4            # 当前 GLB 上限，可改
OTHER_MAX_CONCURRENCY = 8          # 当前非 GLB 上限，可改
POLL_TIMEOUT_SECONDS = 1800        # 30分钟；超时保留 simulation_url，后续 Resume

# =========================
# 各阶段筛选阈值
# =========================
STAGE1_MIN_ABS_SHARPE = 0.80
STAGE1_MIN_ABS_FITNESS = 0.45
STAGE1_EXPLORATION_SHARPE = 0.55
STAGE1_EXPLORATION_FITNESS = 0.25
STAGE1_MIN_POSITIONS = 100

# Stage1 对 PP 只做“潜力门”，允许约 0.8 的信号进入 Stage2；
# Stage2 / Final 再把 PP 本地 Sharpe 预筛提高到 1.0。
STAGE1_PP_SHARPE_GATE = 0.80
FINAL_PP_SHARPE_GATE = 1.00

STAGE2_MIN_ABS_SHARPE = 1.00
STAGE2_MIN_ABS_FITNESS = 0.60
STAGE2_MIN_POSITIONS = 100
STAGE2_FALLBACK_MIN_SELECTED = 3

# =========================
# 各阶段执行开关
# =========================
# 第一次检查 Notebook 时建议先保持 False，确认字段筛选无误后再逐阶段开启。
RUN_STAGE1_CORE = True
RUN_STAGE1_EXTENDED = True
RUN_STAGE2_NEUTRALIZE = True
RUN_STAGE2_GROUP_RANK_FALLBACK = False
RUN_PRE_REPAIR_CHECK = True        # 只 GET /check，不提交 Alpha
RUN_REPAIR = False
RUN_FINAL_CHECK = True             # 只 GET /check，不提交 Alpha

CACHE_DB = str(project_dir / "alpha_results.db")

# 规范化模式并计算当前地区的实际并发数。
TARGET_MODE = normalize_target_mode(TARGET_MODE)
EFFECTIVE_CONCURRENCY = resolve_concurrency(
    REGION,
    CONCURRENCY,
    glb_max=GLB_MAX_CONCURRENCY,
    other_max=OTHER_MAX_CONCURRENCY,
    announce=False,
)

print("Target:", TARGET_MODE)
print("BRAIN:", REGION, UNIVERSE, "D" + str(DELAY), DATASET_ID, NEUTRALIZATION)
print("Data Coverage >=", MIN_DATA_COVERAGE)
print("Concurrency requested/effective:", CONCURRENCY, "/", EFFECTIVE_CONCURRENCY)
print("CACHE_DB:", CACHE_DB)


Target: POWER_POOL_ATOM
BRAIN: GLB TOPDIV3000 D1 risk60 SUBINDUSTRY
Data Coverage >= 0.9
Concurrency requested/effective: 4 / 4
CACHE_DB: F:\一二三\v2\alpha_results.db


## 3. 登录与数据字段筛选

先读取全部字段，再做 **Data Coverage → 人工排除 → MAX_FIELDS**。`MAX_FIELDS` 不会被低 Coverage 字段占掉。


In [3]:
# 登录 BRAIN，后续数据读取和回测请求都复用这个会话。
s = login()
print("BRAIN login: OK")


Loading BRAIN credentials from: F:\一二三\v2\key.txt
Logged in successfully.
BRAIN login: OK


In [ ]:
# 读取当前 Dataset 的全部字段，再依次执行 Coverage、人工排除和数量限制。
# 从 BRAIN 拉取当前 Dataset 的原始 Data Fields。
df_raw = get_datafields(
    s,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    delay=DELAY,
)

# 第一步：先按 Data Coverage 筛选
_df_cov, coverage_report = filter_datafields(
    df_raw,
    min_data_coverage=MIN_DATA_COVERAGE,
    coverage_column=DATA_COVERAGE_COLUMN,
)

# 第二步：再应用人工排除名单
manual_excluded = MANUAL_EXCLUDED_FIELDS_BY_DATASET.get(DATASET_ID, set())
if manual_excluded:
    df_fields = _df_cov[~_df_cov["id"].astype(str).isin(manual_excluded)].copy()
else:
    df_fields = _df_cov.copy()

# 第三步：最后再限制字段数量，避免低 Coverage 字段占名额
if MAX_FIELDS is not None:
    df_fields = df_fields.head(max(0, int(MAX_FIELDS))).copy()

# 整理最终字段表，并为 MATRIX / VECTOR 生成可用于表达式的字段记录。
df_fields = df_fields.reset_index(drop=True)
field_records = prepare_fields(df_fields)

# 输出筛选统计和字段预览，第一次运行时重点检查 Coverage 是否识别正确。
print("Raw Data Fields:", len(df_raw))
print("Coverage column chosen:", coverage_report["coverage_column"])
print("Coverage filtered out:", coverage_report["filtered_out"])
print("Manual excluded after coverage:", len(_df_cov) - len(df_fields) if MAX_FIELDS is None else "see table")
print("Final Data Fields entering Stage1:", len(df_fields))
print("Prepared expressions (VECTOR may x2):", len(field_records))

show_cols = [c for c in (
    "id", "type", "_data_coverage", "dateCoverage", "coverage",
    "pyramidMultiplier", "themes", "dateCreated"
) if c in df_fields.columns]
display(df_fields[show_cols].head(50))

display(pd.DataFrame(field_records)[[
    c for c in (
        "dataset_id", "field", "field_type", "vector_op", "data_coverage",
        "pyramid_multiplier", "expr"
    ) if c in pd.DataFrame(field_records).columns
]].head(20))


## 4. 第一阶段 — 核心搜索

核心算子保持：`raw / rank / zscore / ts_mean / ts_rank / ts_zscore / ts_delta / ts_std_dev`。核心窗口只跑 `5 / 22 / 66`；`120` 不在这一轮。


In [ ]:
# 生成 Stage1 Core 候选，并在回测前完成目标类型标记和上下文校验。
# 使用核心算子与核心窗口生成第一轮候选。
stage1_core_candidates = first_order_candidates(
    field_records,
    ts_operators=CORE_TS_OPS,
    cross_ops=("rank", "zscore"),
    init_decay=INIT_DECAY,
)
stage1_core_candidates = annotate_candidates(stage1_core_candidates, TARGET_MODE)
validate_candidate_context(
    stage1_core_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

print("Stage1 Core candidates:", len(stage1_core_candidates))
print("Has window=120:", any(c.get("window") == 120 for c in stage1_core_candidates))
preview = pd.DataFrame(stage1_core_candidates)
display(preview[[c for c in (
    "dataset_id", "field", "operator", "window", "vector_op",
    "operator_count", "data_field_count", "pp_structure_ok",
    "atom_structure_ok", "expr"
) if c in preview.columns]].head(20))

# 检查 SQLite 中已有缓存，确认哪些候选可直接复用、哪些仍需回测。
stage1_core_resume = resume_summary(
    stage1_core_candidates,
    neutralization=NEUTRALIZATION,
    region=REGION,
    universe=UNIVERSE,
    cache_db=CACHE_DB,
    delay=DELAY,
    truncation=TRUNCATION,
    test_period=TEST_PERIOD,
)


In [ ]:
# 只有打开 Stage1 Core 开关时才真正回测；否则仅生成候选供检查。
if RUN_STAGE1_CORE:
    stage1_core_results = simulate_candidates(
        stage1_core_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage1_core_results = pd.DataFrame()
    print("Stage1 Core 未启动：RUN_STAGE1_CORE=False")


In [ ]:
# 对 Stage1 Core 结果分层，并挑出值得进入定向扩展的字段。
stage1_core_classified = pd.DataFrame()
stage1_extension_fields = []

# 没有结果时不继续扩展；有结果时再进行信号分类。
if stage1_core_results.empty:
    print("暂无 Stage1 Core results。")
else:
    stage1_core_classified = classify_stage1_results(
        stage1_core_results,
        strict_abs_sharpe=STAGE1_MIN_ABS_SHARPE,
        strict_abs_fitness=STAGE1_MIN_ABS_FITNESS,
        min_positions=STAGE1_MIN_POSITIONS,
        exploration_abs_sharpe=STAGE1_EXPLORATION_SHARPE,
        exploration_abs_fitness=STAGE1_EXPLORATION_FITNESS,
    )
    summary = stage1_classification_summary(stage1_core_classified)
    print("Stage1 Core funnel:", summary)

    # 从当前 Candidate 元数据读取字段，避免旧缓存中的元数据污染。
    stage1_core_classified["field"] = stage1_core_classified["candidate"].map(
        lambda c: c.get("field") if isinstance(c, dict) else None
    )
    # 将严格通过、正向、可翻转负向和探索类信号都纳入扩展候选。
    active_mask = (
        stage1_core_classified["is_strict_pass"]
        | stage1_core_classified["is_positive"]
        | stage1_core_classified["is_flippable_negative"]
        | stage1_core_classified["is_exploration"]
    )
    stage1_extension_fields = sorted(
        stage1_core_classified.loc[active_mask, "field"].dropna().astype(str).unique().tolist()
    )
    print("Fields eligible for Extended Stage1:", len(stage1_extension_fields))
    print(stage1_extension_fields)

    top_cols = [c for c in (
        "alpha_id", "field", "signal_class", "sharpe", "fitness", "turnover",
        "positions", "score"
    ) if c in stage1_core_classified.columns]
    display(stage1_core_classified.sort_values("score", ascending=False)[top_cols].head(30))


## 5. 第一阶段 — 定向扩展

只有 Core 已经出现潜力的字段才扩展。加入 `120` 以及 `ts_arg_max / ts_arg_min / ts_quantile`，不会对全部字段重新铺一遍。


In [ ]:
# 只对 Core 阶段已出现潜力的字段生成扩展算子和扩展窗口。
# 为潜力字段生成 120 窗口和扩展算子候选。
stage1_extended_candidates = extended_first_order_candidates(
    field_records,
    active_fields=stage1_extension_fields,
    init_decay=INIT_DECAY,
)
stage1_extended_candidates = annotate_candidates(stage1_extended_candidates, TARGET_MODE)
validate_candidate_context(
    stage1_extended_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

print("Stage1 Extended candidates:", len(stage1_extended_candidates))
# 预览扩展候选，并检查已有缓存命中情况。
if stage1_extended_candidates:
    ext_preview = pd.DataFrame(stage1_extended_candidates)
    display(ext_preview[[c for c in (
        "field", "operator", "window", "search_tier",
        "operator_count", "data_field_count", "expr"
    ) if c in ext_preview.columns]].head(30))
    stage1_extended_resume = resume_summary(
        stage1_extended_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )
else:
    stage1_extended_resume = None


In [ ]:
# 只有开启扩展开关且存在候选时才运行 Stage1 Extended 回测。
if RUN_STAGE1_EXTENDED and stage1_extended_candidates:
    stage1_extended_results = simulate_candidates(
        stage1_extended_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage1_extended_results = pd.DataFrame()
    print("Stage1 Extended 未运行。")


## 6. 第一阶段 — 合并与晋级（支持 1→2→3→6 直达）

缓存恢复、Core/Extended 重建和 Stage1 多样性晋级已全部移入 `machine_lib_V2_1.py`。

即使 `field_records` 不在当前 Kernel 中，第 6 步也会按当前 Dataset / Coverage 参数自动重建字段；只读取 `alpha_results.db`，**不会新建 simulation，也不会 POST**。


In [4]:
# 1236 直达：复杂逻辑已经全部放到 machine_lib_V2_1.py。
stage1_bundle = restore_stage1_cache_and_promote(
    session=s if "s" in globals() else None,
    cache_db=CACHE_DB,
    target_mode=TARGET_MODE,
    dataset_id=DATASET_ID,
    region=REGION,
    universe=UNIVERSE,
    neutralization=NEUTRALIZATION,
    delay=DELAY,
    truncation=TRUNCATION,
    test_period=TEST_PERIOD,
    init_decay=INIT_DECAY,
    min_data_coverage=MIN_DATA_COVERAGE,
    data_coverage_column=DATA_COVERAGE_COLUMN,
    max_fields=MAX_FIELDS,
    manual_excluded_fields_by_dataset=MANUAL_EXCLUDED_FIELDS_BY_DATASET,
    field_records=globals().get("field_records"),
    core_ts_ops=CORE_TS_OPS,
    stage1_min_abs_sharpe=STAGE1_MIN_ABS_SHARPE,
    stage1_min_abs_fitness=STAGE1_MIN_ABS_FITNESS,
    stage1_exploration_sharpe=STAGE1_EXPLORATION_SHARPE,
    stage1_exploration_fitness=STAGE1_EXPLORATION_FITNESS,
    stage1_min_positions=STAGE1_MIN_POSITIONS,
    stage1_pp_sharpe_gate=STAGE1_PP_SHARPE_GATE,
    stage1_keep_per_field=3,
    stage1_min_quality_ratio=0.80,
)

# 恢复后续 Stage2 会继续使用的变量。
field_records = stage1_bundle["field_records"]
stage1_core_results = stage1_bundle["core_results"]
stage1_core_classified = stage1_bundle["core_classified"]
stage1_extension_fields = stage1_bundle["extension_fields"]
stage1_extended_results = stage1_bundle["extended_results"]
stage1_results = stage1_bundle["stage1_results"]
stage1_promoted = stage1_bundle["stage1_promoted"]
stage1_selected = stage1_bundle["stage1_selected"]

if not stage1_selected.empty:
    display(stage1_selected[[c for c in (
        "alpha_id", "dataset_id", "field", "data_coverage",
        "operator", "window", "vector_op",
        "sharpe", "fitness", "turnover", "positions",
        "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "score"
    ) if c in stage1_selected.columns]].head(60))


[429] retry 2/8 | wait 1.0s
Stage1 field source: BRAIN data-fields
Stage1 prepared field expressions: 10
Stage1 Core cache recovery: 150 / 150
Stage1 Core cache missing: 0
Stage1 Core status: {'COMPLETE': 142, 'ERROR': 5, 'RUNNING': 3}
Fields eligible for Extended Stage1: 4
['lending_fee_bid_rate', 'rsk60_crowding', 'rsk60_last', 'rsk60_offer']
Stage1 Extended cache recovery: 72 / 72
Stage1 Extended cache missing: 0
Stage1 Extended status: {'COMPLETE': 72}

Stage1 cache restore / promotion summary
Stage1 total results: 222
Stage1 status: {'COMPLETE': 214, 'ERROR': 5, 'RUNNING': 3}
Stage1 promoted: 11


,alpha_id,dataset_id,field,data_coverage,operator,window,vector_op,sharpe,fitness,turnover,positions,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,score
117,58pJ2zoz,risk60,rsk60_last,0.9503,ts_mean,66.0,vec_sum,-1.53,-2.26,0.0302,1654.0,4,1,True,True,0.987950
147,GrGmZvLG,risk60,rsk60_offer,0.9546,ts_mean,66.0,vec_sum,-1.54,-2.26,0.0284,1620.0,4,1,True,True,0.972185
27,6XpMj6zY,risk60,lending_fee_bid_rate,0.9546,ts_mean,66.0,vec_sum,-1.53,-2.25,0.0283,1621.0,4,1,True,True,0.969820
103,qMN7Ve3E,risk60,rsk60_last,0.9503,ts_std_dev,22.0,vec_avg,-1.59,-1.97,0.0794,1625.0,4,1,True,True,0.929505
122,pwNd3OMo,risk60,rsk60_offer,0.9546,zscore,NaN,vec_avg,-1.44,-2.02,0.0634,1534.0,4,1,True,True,0.879392
2,JjGeodZA,risk60,lending_fee_bid_rate,0.9546,zscore,NaN,vec_avg,-1.42,-1.98,0.0652,1536.0,4,1,True,True,0.867342
107,QPG0g8PK,risk60,rsk60_last,0.9503,zscore,NaN,vec_sum,-1.34,-1.91,0.1187,1621.0,4,1,True,True,0.864527
148,xANvJPnp,risk60,rsk60_offer,0.9546,ts_std_dev,22.0,vec_sum,-1.42,-1.93,0.0760,1534.0,4,1,True,True,0.858896
28,ZYEaARa1,risk60,lending_fee_bid_rate,0.9546,ts_std_dev,22.0,vec_sum,-1.41,-1.92,0.0749,1538.0,4,1,True,True,0.856306
180,ZYEgkGmZ,risk60,rsk60_crowding,0.9512,ts_arg_min,66.0,vec_sum,0.80,0.48,0.0893,1623.0,4,1,True,True,0.827027


## 7. 第二阶段 A — 优先分组中性化

`POWER_POOL_ATOM / ATOM / POWER_POOL` 默认只使用 `sector / industry / subindustry` 三个支持分组，避免无必要引入 `cap / close / volume`。先跑 `group_neutralize`。


In [8]:
# 先对 Stage1 晋级 Alpha 生成 group_neutralize 候选，控制搜索空间。
# 优先只生成 group_neutralize 分支。
stage2_neutral_candidates = second_order_candidates(
    stage1_promoted,
    region=REGION,
    group_ops=("group_neutralize",),
    extended_groups=False,
    target_mode=TARGET_MODE,
)
validate_candidate_context(
    stage2_neutral_candidates,
    dataset_id=DATASET_ID,
    target_mode=TARGET_MODE,
)

# 展示候选结构，并查看这些回测是否已存在于缓存。
print("Stage2 Neutralize candidates:", len(stage2_neutral_candidates))
if stage2_neutral_candidates:
    s2a = pd.DataFrame(stage2_neutral_candidates)
    display(s2a[[c for c in (
        "field", "group_operator", "group", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "expr"
    ) if c in s2a.columns]].head(30))
    stage2_neutral_resume = resume_summary(
        stage2_neutral_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


Stage2 Neutralize candidates: 33


,field,group_operator,group,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,expr
0,rsk60_last,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(sector))"
1,rsk60_last,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(industry))"
2,rsk60_last,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_last), 120), std=4), 66)), densify(subindustry))"
3,rsk60_offer,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(sector))"
4,rsk60_offer,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(industry))"
5,rsk60_offer,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(rsk60_offer), 120), std=4), 66)), densify(subindustry))"
6,lending_fee_bid_rate,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(sector))"
7,lending_fee_bid_rate,group_neutralize,industry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(industry))"
8,lending_fee_bid_rate,group_neutralize,subindustry,6,1,True,True,"group_neutralize(-(ts_mean(winsorize(ts_backfill(vec_sum(lending_fee_bid_rate), 120), std=4), 66)), densify(subindustry))"
9,rsk60_last,group_neutralize,sector,6,1,True,True,"group_neutralize(-(ts_std_dev(winsorize(ts_backfill(vec_avg(rsk60_last), 120), std=4), 22)), densify(sector))"


Resume summary
Candidates: 33
Unique: 33

Completed cache: 33
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 0

Remaining simulations: 0


In [9]:
# 开启对应开关后运行 Stage2 Neutralize 回测，否则保持为空。
if RUN_STAGE2_NEUTRALIZE and stage2_neutral_candidates:
    stage2_neutral_results = simulate_candidates(
        stage2_neutral_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage2_neutral_results = pd.DataFrame()
    print("Stage2 Neutralize 未运行。")


Resume summary
Candidates: 33
Unique: 33

Completed cache: 33
Submitted / running: 0
Errors to retry: 0
Auth errors to retry: 0
Invalid: 0
Uncertain submission (not auto-retried): 0
New: 0

Remaining simulations: 0
Requested concurrency: 4
Region concurrency limit: 4
Effective concurrency: 4
Polling timeout per worker: 1800s
[001/33] CACHE COMPLETE
[002/33] CACHE COMPLETE
[003/33] CACHE COMPLETE
[004/33] CACHE COMPLETE
[005/33] CACHE COMPLETE
[006/33] CACHE COMPLETE
[007/33] CACHE COMPLETE
[008/33] CACHE COMPLETE
[009/33] CACHE COMPLETE
[010/33] CACHE COMPLETE
[011/33] CACHE COMPLETE
[012/33] CACHE COMPLETE
[013/33] CACHE COMPLETE
[014/33] CACHE COMPLETE
[015/33] CACHE COMPLETE
[016/33] CACHE COMPLETE
[017/33] CACHE COMPLETE
[018/33] CACHE COMPLETE
[019/33] CACHE COMPLETE
[020/33] CACHE COMPLETE
[021/33] CACHE COMPLETE
[022/33] CACHE COMPLETE
[023/33] CACHE COMPLETE
[024/33] CACHE COMPLETE
[025/33] CACHE COMPLETE
[026/33] CACHE COMPLETE
[027/33] CACHE COMPLETE
[028/33] CACHE COMPLETE
[

In [10]:
# 对 Neutralize 结果执行 Stage2 晋级筛选，保留每个字段的优质候选。
# 对 Neutralize 分支进行 Stage2 质量筛选。
stage2_neutral_promoted, stage2_neutral_selected = promote_candidates_for_target(
    stage2_neutral_results,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=2,
    max_total=40,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)
print("Stage2 Neutralize selected:", len(stage2_neutral_selected))
if not stage2_neutral_selected.empty:
    display(stage2_neutral_selected[[c for c in (
        "alpha_id", "field", "group", "sharpe", "fitness", "turnover",
        "positions", "operator_count", "data_field_count", "score"
    ) if c in stage2_neutral_selected.columns]].head(40))


Stage2 Neutralize selected: 6


,alpha_id,field,group,sharpe,fitness,turnover,positions,operator_count,data_field_count,score
0,vRNZO5eQ,rsk60_last,sector,1.53,2.26,0.0302,1655,6,1,0.899242
4,RRm9EQZb,rsk60_offer,industry,1.54,2.26,0.0283,1620,6,1,0.890909
3,pwNxvzoo,rsk60_offer,sector,1.54,2.26,0.0284,1620,6,1,0.886364
6,xANVrqXl,lending_fee_bid_rate,sector,1.53,2.25,0.0283,1621,6,1,0.866667
1,P0GxMn7p,rsk60_last,industry,1.52,2.25,0.0301,1655,6,1,0.858333
7,XgoVRMoz,lending_fee_bid_rate,industry,1.52,2.23,0.0282,1621,6,1,0.833333


## 8. 第二阶段 B — Group Rank 补充

只有 Neutralize 产出的可用候选少于 `STAGE2_FALLBACK_MIN_SELECTED` 时才生成 `group_rank`，保留这个算子但不默认全铺。


In [11]:
# 当 Neutralize 候选不足时，才启用 group_rank 作为补充搜索。
# 根据 Neutralize 已选数量判断是否需要启用补充分支。
need_group_rank_fallback = len(stage2_neutral_selected) < STAGE2_FALLBACK_MIN_SELECTED

# 只有候选不足时才生成 group_rank，避免默认全铺。
if need_group_rank_fallback:
    stage2_rank_candidates = second_order_candidates(
        stage1_promoted,
        region=REGION,
        group_ops=("group_rank",),
        extended_groups=False,
        target_mode=TARGET_MODE,
    )
    validate_candidate_context(
        stage2_rank_candidates,
        dataset_id=DATASET_ID,
        target_mode=TARGET_MODE,
    )
else:
    stage2_rank_candidates = []

print("Need group_rank fallback:", need_group_rank_fallback)
print("Stage2 Rank candidates:", len(stage2_rank_candidates))
if stage2_rank_candidates:
    stage2_rank_resume = resume_summary(
        stage2_rank_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


Need group_rank fallback: False
Stage2 Rank candidates: 0


In [ ]:
# 仅在确实需要且开关打开时运行 Group Rank 回测。
if RUN_STAGE2_GROUP_RANK_FALLBACK and stage2_rank_candidates:
    stage2_rank_results = simulate_candidates(
        stage2_rank_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    stage2_rank_results = pd.DataFrame()
    print("Stage2 Group Rank 未运行。")


## 9. 第二阶段 — 合并与最终预筛


In [12]:
# 兼容跳过 Stage2 Group Rank 的情况
if "stage2_neutral_results" not in globals():
    stage2_neutral_results = pd.DataFrame()

if "stage2_rank_results" not in globals():
    stage2_rank_results = pd.DataFrame()
# 合并 Stage2 两条分支结果，去重并生成 Repair 前的正式候选。
# 合并 Neutralize 与 Rank 两条 Stage2 分支。
stage2_frames = [x for x in (stage2_neutral_results, stage2_rank_results) if not x.empty]
if stage2_frames:
    stage2_results = pd.concat(stage2_frames, ignore_index=True, sort=False)
    if "sim_key" in stage2_results.columns:
        stage2_results = stage2_results.drop_duplicates(subset=["sim_key"], keep="last")
else:
    stage2_results = pd.DataFrame()

# 按目标模式进行最终 Stage2 晋级筛选。
stage2_promoted, stage2_selected = promote_candidates_for_target(
    stage2_results,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=2,
    max_total=40,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)
validate_candidate_context(stage2_promoted, dataset_id=DATASET_ID, target_mode=TARGET_MODE)

print("Stage2 total results:", len(stage2_results))
print("Stage2 selected:", len(stage2_selected))
if not stage2_selected.empty:
    display(stage2_selected[[c for c in (
        "alpha_id", "dataset_id", "field", "group_operator", "group", "sharpe",
        "fitness", "turnover", "positions", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "score"
    ) if c in stage2_selected.columns]].head(40))


Stage2 total results: 33
Stage2 selected: 6


,alpha_id,dataset_id,field,group_operator,group,sharpe,fitness,turnover,positions,operator_count,data_field_count,pp_structure_ok,atom_structure_ok,score
0,vRNZO5eQ,risk60,rsk60_last,group_neutralize,sector,1.53,2.26,0.0302,1655,6,1,True,True,0.899242
4,RRm9EQZb,risk60,rsk60_offer,group_neutralize,industry,1.54,2.26,0.0283,1620,6,1,True,True,0.890909
3,pwNxvzoo,risk60,rsk60_offer,group_neutralize,sector,1.54,2.26,0.0284,1620,6,1,True,True,0.886364
6,xANVrqXl,risk60,lending_fee_bid_rate,group_neutralize,sector,1.53,2.25,0.0283,1621,6,1,True,True,0.866667
1,P0GxMn7p,risk60,rsk60_last,group_neutralize,industry,1.52,2.25,0.0301,1655,6,1,True,True,0.858333
7,XgoVRMoz,risk60,lending_fee_bid_rate,group_neutralize,industry,1.52,2.23,0.0282,1621,6,1,True,True,0.833333


## 10. Repair 前实时提交检查

这里调用 `/alphas/{id}/check`，**只检查，不提交**。Repair 优先根据真实 FAIL 原因生成少量变体。


In [13]:
# 对 Stage2 候选调用实时 /check，只检查提交条件，不执行提交。
# 开关开启且存在候选时，获取平台真实提交检查结果。
if RUN_PRE_REPAIR_CHECK and not stage2_selected.empty:
    pre_repair_checks = check_submission_candidates(
        s,
        stage2_selected,
        limit=30,
        cache_db=CACHE_DB,
    )
else:
    pre_repair_checks = pd.DataFrame()
    print("Pre-repair check 未运行。")

# 展示通过数量和每条 Alpha 的失败原因，供 Repair 使用。
if not pre_repair_checks.empty:
    print("Pre-repair PASS:", int(pre_repair_checks["check_pass"].fillna(False).sum()))
    display(pre_repair_checks[[c for c in (
        "alpha_id", "field", "sharpe", "fitness", "turnover",
        "check_pass", "self_correlation", "failed_check_names", "check_error"
    ) if c in pre_repair_checks.columns]].head(30))


Pre-repair PASS: 0


,alpha_id,field,sharpe,fitness,turnover,check_pass,self_correlation,failed_check_names,check_error
0,vRNZO5eQ,rsk60_last,1.53,2.26,0.0302,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
1,RRm9EQZb,rsk60_offer,1.54,2.26,0.0283,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
2,pwNxvzoo,rsk60_offer,1.54,2.26,0.0284,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
3,xANVrqXl,lending_fee_bid_rate,1.53,2.25,0.0283,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
4,P0GxMn7p,rsk60_last,1.52,2.25,0.0301,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None
5,XgoVRMoz,lending_fee_bid_rate,1.52,2.23,0.0282,False,None,"[LOW_SHARPE, LOW_GLB_EMEA_SHARPE, LOW_GLB_APAC_SHARPE, LOW_SUB_UNIVERSE_SHARPE]",None


## 11. 按失败原因定向 Repair

有 live check 时按 `LOW_SHARPE / LOW_2Y_SHARPE / LOW_SUB_UNIVERSE_SHARPE / CONCENTRATED_WEIGHT / HIGH_TURNOVER` 触发；没有 check 时才回退旧版高换手 Repair。`ATOM / PP+ATOM` 不加入跨 Dataset 的 region event `trade_when`。


In [ ]:
# 优先根据真实失败原因生成 Repair；没有实时检查结果时才回退旧逻辑。
# 有真实 check 结果时优先使用 Fail-aware Repair。
if not pre_repair_checks.empty:
    repair_candidates = submission_aware_repair_candidates(
        pre_repair_checks,
        region=REGION,
        target_mode=TARGET_MODE,
        max_variants_per_parent=6,
        turnover_trigger=0.35,
    )
elif not stage2_selected.empty:
    repair_candidates = targeted_repair_candidates(
        stage2_selected,
        region=REGION,
        max_variants_per_parent=6,
        turnover_trigger=0.35,
    )
    repair_candidates = annotate_candidates(repair_candidates, TARGET_MODE)
else:
    repair_candidates = []

# Repair 生成后再次校验 Dataset 与目标上下文，防止串数据。
validate_candidate_context(repair_candidates, dataset_id=DATASET_ID, target_mode=TARGET_MODE)
print("Repair candidates:", len(repair_candidates))
if repair_candidates:
    rp = pd.DataFrame(repair_candidates)
    display(rp[[c for c in (
        "field", "repair", "decay", "operator_count", "data_field_count",
        "pp_structure_ok", "atom_structure_ok", "expr"
    ) if c in rp.columns]].head(30))
    repair_resume = resume_summary(
        repair_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        cache_db=CACHE_DB,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
    )


In [ ]:
# 只有打开 Repair 开关且存在候选时才运行修复回测。
if RUN_REPAIR and repair_candidates:
    repair_results = simulate_candidates(
        repair_candidates,
        neutralization=NEUTRALIZATION,
        region=REGION,
        universe=UNIVERSE,
        session=s,
        cache_db=CACHE_DB,
        concurrency=CONCURRENCY,
        delay=DELAY,
        truncation=TRUNCATION,
        test_period=TEST_PERIOD,
        progress_every=10,
        poll_timeout_seconds=POLL_TIMEOUT_SECONDS,
        glb_max_concurrency=GLB_MAX_CONCURRENCY,
        other_max_concurrency=OTHER_MAX_CONCURRENCY,
    )
else:
    repair_results = pd.DataFrame()
    print("Repair 未运行。")


## 12. 最终候选

最终只保留当前目标模式的本地结构/质量预筛候选；平台 `/check` 仍是最终裁决。


In [ ]:
# 合并 Stage2 与 Repair 结果，形成当前目标模式下的最终候选池。
# 收集 Stage2 与 Repair 两部分可用结果。
combined_final_frames = []
if not stage2_selected.empty:
    combined_final_frames.append(stage2_selected)
if not repair_results.empty:
    combined_final_frames.append(repair_results)

if combined_final_frames:
    final_pool = pd.concat(combined_final_frames, ignore_index=True, sort=False)
    if "sim_key" in final_pool.columns:
        final_pool = final_pool.drop_duplicates(subset=["sim_key"], keep="last")
else:
    final_pool = pd.DataFrame()

# 进行最终本地预筛；平台实时 /check 仍是最终裁决。
_final_promoted, final_results = promote_candidates_for_target(
    final_pool,
    target_mode=TARGET_MODE,
    min_abs_sharpe=STAGE2_MIN_ABS_SHARPE,
    min_abs_fitness=STAGE2_MIN_ABS_FITNESS,
    min_positions=STAGE2_MIN_POSITIONS,
    keep_per_field=4,
    max_total=100,
    allow_negative_flip=False,
    power_pool_min_sharpe=FINAL_PP_SHARPE_GATE,
)

print("Final candidates:", len(final_results))
# 给候选标记本地结构类型，便于区分 PP、ATOM 和普通研究候选。
if not final_results.empty:
    def local_type(c):
        if not isinstance(c, dict):
            return "UNKNOWN"
        if c.get("pp_atom_structure_ok"):
            return "PP + ATOM structure"
        if c.get("pp_structure_ok"):
            return "Power Pool structure"
        if c.get("atom_structure_ok"):
            return "ATOM structure"
        return "Regular / research"

    final_results = final_results.copy()
    final_results["local_type"] = final_results["candidate"].map(local_type)
    display(final_results[[c for c in (
        "alpha_id", "dataset_id", "field", "local_type", "data_coverage",
        "sharpe", "fitness", "turnover", "positions", "operator_count",
        "data_field_count", "pyramid_multiplier", "score"
    ) if c in final_results.columns]].head(100))


## 13. 最终提交检查

仍然**不会自动提交 Alpha**。这里把 PASS 数量和失败原因结构化保存。


In [ ]:
# 对最终候选再次执行平台实时检查，并统计主要失败原因。
# 开关开启时对最终候选执行实时提交检查。
if RUN_FINAL_CHECK and not final_results.empty:
    final_checks = check_submission_candidates(
        s,
        final_results,
        limit=100,
        cache_db=CACHE_DB,
    )
else:
    final_checks = pd.DataFrame()
    print("Final check 未运行。")

if not final_checks.empty:
    pass_count = int(final_checks["check_pass"].fillna(False).sum())
    print("Final CHECK PASS:", pass_count, "/", len(final_checks))
    display(final_checks[[c for c in (
        "alpha_id", "field", "local_type", "sharpe", "fitness", "turnover",
        "check_pass", "self_correlation", "failed_check_names", "check_error"
    ) if c in final_checks.columns]].head(100))

    # 汇总失败类型，后续可用于优化 Repair 和搜索策略。
    failed_names = (
        final_checks.loc[~final_checks["check_pass"].fillna(False), "failed_check_names"]
        .explode()
        .dropna()
        .astype(str)
        .value_counts()
    )
    if not failed_names.empty:
        print("\nFail reason counts:")
        display(failed_names.rename("count").to_frame())


## 14. 结果导出与上下文汇总

Simulation 仍在原 `alpha_results` 表；V2.1 另外在同一个 DB 的 `alpha_contexts` 表保存 Dataset / Target / Stage 等研究上下文，不改变旧缓存主键。


In [ ]:
# 将各阶段非空结果导出为 CSV，并输出当前 Dataset/Target 的上下文统计。
# 创建本轮结果目录。
OUTPUT_DIR = project_dir / "results_v2_1"
OUTPUT_DIR.mkdir(exist_ok=True)

exports = {
    "data_fields_filtered.csv": df_fields,
    "stage1_core_results.csv": stage1_core_results,
    "stage1_extended_results.csv": stage1_extended_results,
    "stage1_results.csv": stage1_results,
    "stage2_neutral_results.csv": stage2_neutral_results,
    "stage2_rank_results.csv": stage2_rank_results,
    "stage2_results.csv": stage2_results,
    "pre_repair_checks.csv": pre_repair_checks,
    "repair_results.csv": repair_results,
    "final_results.csv": final_results,
    "final_checks.csv": final_checks,
}

# 只导出非空结果表，避免生成无意义的空文件。
for filename, frame in exports.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        path = OUTPUT_DIR / filename
        frame.to_csv(path, index=False)
        print("Exported:", path)

print("\nCurrent Dataset / Target contexts:")
display(context_summary(CACHE_DB, dataset_id=DATASET_ID, target_mode=TARGET_MODE))


## 15. 缓存与维护

全局 simulation cache 与当前研究 context 分开看，避免把不同 Dataset 的总数误认为当前 Run。


In [ ]:
# 分开查看全局回测缓存和当前 Dataset/Target 的研究上下文。
print("Global simulation cache:")
cache_summary(CACHE_DB)

print("\nCurrent Dataset / Target contexts:")
display(context_summary(CACHE_DB, dataset_id=DATASET_ID, target_mode=TARGET_MODE))
